# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aun-Mehdi117/-Flyrank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

FlyRank's editors can't review 30,000 pages by hand, but a plain rule-of-thumb ("fix
whatever's declining") is too blunt: it ignores whether the page still has demand, whether
it's actually stale, and whether the drop is real or just noise. I want to turn "which page
should I look at first?" into a ranked, explainable queue instead of a gut call. This lane also
has the clearest existing scaffolding in the starter pipeline (baseline score, reason codes,
model comparison already implemented in `scripts/01-05`), so I can spend my 7 weeks improving
and validating a method rather than inventing plumbing from scratch. I'll confirm or swap this
by end of Week 4 if the signal audit says otherwise.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Decision:** Given a client's content inventory, which pages should a content editor review
first for refresh, expansion, protection, or pruning — out of far more candidates than the
team has time to check by hand?

**Who acts, and how:** A content editor (or SEO strategist) with a fixed weekly review
capacity (say, the top 20-50 pages) works down my ranked queue instead of picking pages
randomly or purely by newest/oldest.

**Cost of a wrong call, in both directions:**
- *False positive* (I flag a page as a priority and it wasn't): wasted editor hours on a page
  that didn't need attention — the review capacity is scarce, so this has a real opportunity
  cost, since a page that *did* need help waited another week.
- *False negative* (a genuinely declining, high-demand page never surfaces): the page keeps
  losing visibility unnoticed, and by the time anyone spots it manually the drop may be harder
  to reverse. This is the more expensive error, since it's silent.

Because false negatives are costlier than false positives here, I'll care more about recall at
the top of the queue (am I missing real opportunities?) than about raw accuracy.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Numbers pulled straight from `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32
pseudonymized clients):

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
n = len(df)
print(f"rows: {n}, unique clients: {df.client_id.nunique()}")

# candidates for two of the starter's own reason codes
declining_with_demand = df[(df.trend_direction == "down") & (df.impressions_90d >= 100)]
page_one_decay_risk = df[(df.avg_position > 0) & (df.avg_position <= 10) & (df.content_age_days >= 180)]
low_ctr_visible = df[(df.impressions_90d >= 500) & (df.avg_position.between(1, 20, inclusive="both")) & (df.ctr < 0.5)]

print(f"declining_with_demand (down & impressions_90d>=100): {len(declining_with_demand)} rows "
      f"({len(declining_with_demand)/n:.1%})")
print(f"page_one_decay_risk (top-10 position & 180+ days old): {len(page_one_decay_risk)} rows "
      f"({len(page_one_decay_risk)/n:.1%})")
print(f"low_ctr_visible_page (500+ impressions, pos 1-20, ctr<0.5%): {len(low_ctr_visible)} rows "
      f"({len(low_ctr_visible)/n:.1%})")


rows: 30000, unique clients: 32
declining_with_demand (down & impressions_90d>=100): 13152 rows (43.8%)
page_one_decay_risk (top-10 position & 180+ days old): 7076 rows (23.6%)
low_ctr_visible_page (500+ impressions, pos 1-20, ctr<0.5%): 9745 rows (32.5%)


**What these numbers say about the lane:**

- **43.8%** of rows (13,152 of 30,000) are "declining with demand" — a page losing ground that
  still has real impressions. That's far too many for manual review, and it's the core
  population a ranking model would prioritize *within*.
- **23.6%** of rows (7,076) sit in the top 10 positions but are 180+ days old — a real "decay
  risk" pool worth distinguishing from pages that are simply young and new.
- **32.5%** of rows (9,745) have decent visibility (500+ impressions, top-20 position) but a
  sub-0.5% CTR — a candidate pool for title/snippet review, distinct from the decline pool.

These three groups overlap only partially, which is exactly why a single rule ("fix whatever's
declining") isn't precise enough: there's real structure here for a ranked score to separate,
not just one obvious bucket. That's the case for spending 7 weeks on this instead of writing
one if-statement.

## 4. Careful words: what I can and can't claim

**What I can say, once this is built:**
- *Observed*: "pages with these characteristics historically declined / carried demand /
  under-captured clicks at these rates" — a description of what happened in the data.
- *Directional*: "pages with more of signal X tend to also show pattern Y" — a correlation,
  not a mechanism.
- *Decision-support*: "here is a ranked queue an editor can use to spend limited review time
  more efficiently than reviewing randomly or by recency" — a tool to prioritize attention,
  not an oracle.

**What I will never claim:**
- That a refresh *caused* a recovery — I have no experiment or causal design here, only
  observational data. I can say a page recovered after a refresh, never that the refresh
  is why.
- That I predicted or reverse-engineered a Google ranking factor. I only have FlyRank's own
  observed signals (impressions, clicks, position, CTR, engagement) — not the algorithm.
- That a "declining" label is certain rather than a threshold I chose (see Section 7 of the
  lane guide: real decline vs. consolidation, seasonality, SERP change, or plain noise) — I'll
  need to rule those look-alikes out before trusting any label built from `trend_direction`.
- That a wrong rank on my queue is free — Section 2 already prices that in.

In [4]:
# Sanity check: the label trap flagged in the flyrank-data skill.
# is_declining_label / trend_direction is DERIVED from trend_pct, so trend_pct
# (and trend_direction) can never be used as a feature later - only as a target/proxy.
print("trend_direction value counts:")
print(df.trend_direction.value_counts())
print()
print("Any content with trend_pct but no trend_direction (should be none):",
      ((df.trend_pct.notna()) & (df.trend_direction.isna())).sum())


trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Any content with trend_pct but no trend_direction (should be none): 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.